# LangGraph, Part 1 — State, Nodes & Edges
### Module 6 — Stateful Agents (Part 1 of 2)

**Goal:** build your first **LangGraph** agent. Where a LangChain chain runs straight through, a **graph** has *state*, *nodes* (steps), and *edges* (flow) — and can branch and loop. We build a simple **malicious-URL analyzer**, closely following Omar Santos's `basic_example.py`.

> **Sources:** Omar Santos — repo `part5_agents_and_tools/agent_deep_dive/langgraph`; book Ch. 4; LinkedIn course Section 4.

> **Setup:** `pip install langgraph langchain-openai`. The LLM node is guarded on `OPENAI_API_KEY` (no key → it returns a stub so the graph still runs). Ships without outputs.

## 🛠️ Setup

In [ ]:
# pip install langgraph langchain-openai python-dotenv
import os
try:
    from dotenv import load_dotenv, find_dotenv; load_dotenv(find_dotenv())
except Exception:
    pass
HAS_KEY = bool(os.getenv("OPENAI_API_KEY"))
print("OPENAI_API_KEY found:", HAS_KEY)

# 🧭 1 — Why a Graph? (vs. a Chain)

- A LangChain chain runs **straight through**: A → B → C
- Real agents need to **remember**, **branch**, and **loop**
- LangGraph adds **state** (shared memory) + **nodes** + **edges**
- You draw the workflow as a graph and let it run

> ### 🎤 Instructor Script
>
> In Module 5 a chain ran straight through: prompt, model, parser. But a real agent needs to keep notes as it works, choose different paths, and sometimes loop back. LangGraph gives you that: a shared state object that every step can read and write, nodes that are just functions doing one step, and edges that decide what runs next. You literally describe the workflow as a graph. We start with the simplest possible one and grow it in Part 2.

# 📦 2 — Define the State

- State = the graph's shared memory (a `TypedDict`)
- Every node receives it and returns an updated copy
- Here: the URL, the analysis results, and the verdict

> ### 🎤 Instructor Script
>
> First we define the state — the graph's memory. It's a typed dictionary that gets passed from node to node; each node reads what it needs and returns updates. For our URL analyzer the state holds the URL we're examining, any analysis results we gather, and the final verdict. Keeping all shared data in one state object is what lets later nodes build on earlier ones.

In [ ]:
from typing import TypedDict

class AgentState(TypedDict):
    url: str                # the URL to analyze
    features: dict          # signals we extract about it
    verdict: str            # final "malicious" / "benign" + reason

# 🔧 3 — Define Nodes (the steps)

- A node is a function: takes the state, returns updates
- Node 1: a tool that extracts features from the URL (no LLM)
- Node 2: an LLM decides malicious vs. benign from those features
- Nodes do one job each — easy to read and test

> ### 🎤 Instructor Script
>
> Nodes are the building blocks — plain Python functions that take the state and return the fields they changed. Our first node is a mock analysis tool that pulls simple features out of the URL, like whether it uses an IP address or suspicious words; no AI needed. The second node asks the LLM to weigh those features and return a verdict. We guard the LLM so the graph still runs without a key. Notice each node does exactly one thing.

In [ ]:
def extract_features(state: AgentState) -> dict:
    url = state["url"]
    features = {
        "has_ip": any(ch.isdigit() for ch in url.split("/")[2]) if "//" in url else False,
        "suspicious_words": [w for w in ["login", "verify", "free", "bank", "update"] if w in url.lower()],
        "length": len(url),
    }
    print("  [node] extract_features ->", features)
    return {"features": features}

def llm_verdict(state: AgentState) -> dict:
    prompt = f"URL: {state['url']}\nFeatures: {state['features']}\nIs this malicious or benign? One sentence."
    if HAS_KEY:
        from langchain_openai import ChatOpenAI
        text = ChatOpenAI(model="gpt-4o-mini", temperature=0).invoke(prompt).content
    else:
        text = "[no key] benign — stub verdict (set OPENAI_API_KEY for a real call)."
    print("  [node] llm_verdict ->", text)
    return {"verdict": text}

# 🔗 4 — Build & Run the Graph

- `StateGraph(AgentState)` — create the graph over our state
- `add_node(...)` for each step; `add_edge(...)` to connect them
- Set the entry point; end with `END`
- `.compile()` then `.invoke({...})` to run it

> ### 🎤 Instructor Script
>
> Now we wire the nodes into a graph. We create a StateGraph over our state type, add the two nodes, and connect them with edges: start at feature extraction, then go to the verdict, then to END. We set the entry point, compile the graph, and invoke it with a URL. Watch the nodes print as the state flows through. This tiny two-node graph is the same machinery that powers complex agents — you just add more nodes and smarter edges, which is Part 2.

In [ ]:
from langgraph.graph import StateGraph, END

builder = StateGraph(AgentState)
builder.add_node("extract", extract_features)
builder.add_node("verdict", llm_verdict)
builder.set_entry_point("extract")
builder.add_edge("extract", "verdict")
builder.add_edge("verdict", END)
graph = builder.compile()

result = graph.invoke({"url": "http://192.168.0.5/login/verify-account", "features": {}, "verdict": ""})
print("\nFINAL VERDICT:", result["verdict"])

# ✅ Summary — Part 1

- A LangGraph agent = **state** + **nodes** + **edges**
- Nodes are functions that read/update the shared state
- Edges define the flow; `END` stops it
- You built and ran a 2-node URL-analysis agent
- Next (Part 2): branching, tools, and memory

> ### 🎤 Instructor Script
>
> You've built your first LangGraph agent: a shared state, two nodes, edges connecting them, and a compiled graph you invoked on a URL. That's the whole mental model — everything fancier is more nodes and smarter edges. In Part 2 we add conditional edges so the agent chooses its own path, give it real tools to call, and add memory so it can hold a conversation — turning this into a proper ethical-hacking assistant.

In [ ]:
print("Part 1 done: you can define state, write nodes, connect edges, and run a graph.")
print("Next: langgraph_part2.ipynb -> branching, tools, and memory.")